# Time Series SSL Benchmark — ETTm1

Trains and evaluates three self-supervised models on **ETTm1** (electricity transformer, 7 variables, 15-min intervals):

| Model | Method |
|-------|--------|
| **DINO** | DWT student-teacher (soft-threshold teacher, zero-out / noise student) |
| **Discrete JEPA** | VQ-based joint-embedding predictive architecture |
| **PatchTST** | Masked patch self-supervised |

Each model: **pretrain → forecasting downstream**.

---
**Quick-test tip** — reduce epochs before running:
```
DINO : TSDINOALT 4/config.py          → "epochs": 20,  "saveckp_freq": 5
JEPA : Descrete_JEPA/config_files/config_pretrain.py  → "num_epochs": 100
```

## 1 — Mount Drive & set project root

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# ── Change this to wherever you uploaded the allthree folder ─────────────────
PROJECT_ROOT = '/content/drive/MyDrive/allthree'

os.chdir(PROJECT_ROOT)
print('Working directory:', os.getcwd())
print('Contents:', os.listdir('.'))

## 1b — Choose dataset

In [ ]:
# ── Set datasets here ─────────────────────────────────────────────────────────
# Options: "ettm1"  "etth1"  "etth2"  "ettm2"  "weather"  "electricity"  "traffic"

PRETRAIN_DATASET  = "ettm1"   # dataset used for self-supervised pretraining
FORECAST_DATASET  = "ettm1"   # dataset used for forecasting downstream
                               # set to a different value to pretrain on one, forecast on another

# Quick sanity-check: confirm the CSVs exist
import pandas as pd, sys, os
sys.path.insert(0, os.getcwd())           # make dataset_registry importable
from dataset_registry import get_dataset_info

for label, name in [("Pretrain", PRETRAIN_DATASET), ("Forecast", FORECAST_DATASET)]:
    ds = get_dataset_info(name)
    df = pd.read_csv(ds["csv_path"], nrows=0)
    print(f"{label:8s} → {name:12s}  {ds['c_in']} vars   {ds['csv_path']}")

## 2 — Install dependencies

In [ ]:
!pip install PyWavelets --quiet

import torch, sklearn, pywt, numpy, pandas
print(f'torch   {torch.__version__}')
print(f'sklearn {sklearn.__version__}')
print(f'pywt    {pywt.__version__}')
print(f'GPU     {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "not available"}')

## 3 — Verify ETTm1 data

In [ ]:
import pandas as pd

for label, name in [("Pretrain", PRETRAIN_DATASET), ("Forecast", FORECAST_DATASET)]:
    ds = get_dataset_info(name)
    df = pd.read_csv(ds["csv_path"])
    print(f"── {label}: {name} ──")
    print(f"  Columns : {df.columns.tolist()}")
    print(f"  Rows    : {len(df)}")
    print(df.head(2))

---
## 4 — Run DINO (pretrain + forecasting)

In [ ]:
from Train_and_downstream import run
run(model='dino', pretrain_dataset=PRETRAIN_DATASET, forecast_dataset=FORECAST_DATASET, skip_train=False)

---
## 5 — Run Discrete JEPA (pretrain + forecasting)

In [ ]:
from Train_and_downstream import run
run(model='jepa', pretrain_dataset=PRETRAIN_DATASET, forecast_dataset=FORECAST_DATASET, skip_train=False)

---
## 6 — Run PatchTST (pretrain + forecasting finetune)

In [ ]:
from Train_and_downstream import run
run(model='patchtst', skip_train=False)

---
## 7 — (Optional) Run all three back-to-back

In [ ]:
# Uncomment to run everything in one shot
# from Train_and_downstream import run
# run(model='dino',     pretrain_dataset=PRETRAIN_DATASET, forecast_dataset=FORECAST_DATASET, skip_train=False)
# run(model='jepa',     pretrain_dataset=PRETRAIN_DATASET, forecast_dataset=FORECAST_DATASET, skip_train=False)
# run(model='patchtst', skip_train=False)   # PatchTST always uses ettm1

---
## 8 — Sanity checks (run these first if something breaks)

In [ ]:
# ── DINO DataPuller on pretrain dataset ───────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath('TSDINOALT 4'))
import dataPuller as dp

ds_pre_info = get_dataset_info(PRETRAIN_DATASET)

ds_obj = dp.DataPuller(
    data_dir  = ds_pre_info["csv_path"],
    split     = 'train',
    batch_size= 32,
    patch_size= 12,
    step_size = 12,
)
sample = ds_obj[0]
print('Dataset length :', len(ds_obj))
print('Sample shape   :', sample.shape)
print('mean / std     : {:.4f} / {:.4f}'.format(sample.mean().item(), sample.std().item()))

In [ ]:
# ── DWT augmentation pipeline ─────────────────────────────────────────────────
from data_agumentation import DWTAugmentation
import torch

x = torch.randn(48, 7)
for mode, kwargs in [
    ('soft_threshold',  {'soft_threshold_sigma': 0.3}),
    ('zero_out_detail', {'zero_out_ratio': 0.3, 'finest_levels': 1}),
    ('high_perturb',    {'high_perturb_noise_range': (0.03, 0.08)}),
    ('low_pass',        {}),
]:
    out = DWTAugmentation(mode=mode, **kwargs)(x)
    print(f'DWT {mode:18s}  in {tuple(x.shape)}  out {tuple(out.shape)}  '
          f'mean {out.mean():.4f}  std {out.std():.4f}')

In [ ]:
# ── Forecasting data loaders ──────────────────────────────────────────────────
ds_fore_info = get_dataset_info(FORECAST_DATASET)

train_fc = dp.DataPullerForecastingTrain(
    data_dir  = ds_fore_info["csv_path"],
    split     = 'train',
    batch_size= 32,
    patch_size= 12,
    pred_len  = 96,
    var_list  = ds_fore_info["columns"],
)
x_in, y_out = train_fc[0]
print(f'Train forecasting  x:{tuple(x_in.shape)}  y:{tuple(y_out.shape)}')
print(f'Dataset length: {len(train_fc)}')